In [2]:
import pandas as pd

In [3]:
dfOrders = pd.read_csv("olist_orders_dataset.csv", encoding="latin1") 


In [4]:
dfCustomers = pd.read_csv("olist_customers_dataset.csv", encoding="latin1") 
dfOrderItems = pd.read_csv("olist_order_items_dataset.csv", encoding="latin1")
dfProducts = pd.read_csv("olist_products_dataset.csv", encoding="latin1")
dfPayments = pd.read_csv("olist_order_payments_dataset.csv", encoding="latin1")
dfReviews = pd.read_csv("olist_order_reviews_dataset.csv", encoding="latin1")
dfSellers = pd.read_csv("olist_sellers_dataset.csv", encoding="latin1")

In [5]:
dfML = dfOrders.copy()

In [6]:
# merging customers into orders table
customer_columns = [
    "customer_id",
    "customer_city",
    "customer_state"
]

dfML = dfML.merge(
    dfCustomers[customer_columns],
    on="customer_id",
    how="left"
)

In [7]:
dfML.shape

(99441, 10)

In [8]:
# merging reviews table
review_columns = [
    "order_id",
    "review_score"
]

dfML = dfML.merge(
    dfReviews[review_columns],
    on="order_id",
    how="left"
)

In [10]:
#aggregating payments before merging
payment_summary = (
    dfPayments
    .groupby("order_id")
    .agg(
        payment_value=("payment_value", "sum"),
        payment_installments=("payment_installments", "max"),
        payment_type=("payment_type", "first")
    )
    .reset_index()
)
dfML = dfML.merge(
    payment_summary,
    on="order_id",
    how="left"
)

In [11]:
#aggregating and merging order items table
order_summary = (
    dfOrderItems
    .groupby("order_id")
    .agg(
        number_of_items=("order_item_id", "count"),
        total_product_price=("price", "sum"),
        freight_value=("freight_value", "sum")
    )
    .reset_index()
)

dfML = dfML.merge(
    order_summary,
    on="order_id",
    how="left"
)

In [12]:
items_products = dfOrderItems.merge(
    dfProducts,
    on="product_id",
    how="left"
)
items_products["product_volume"] = (
    items_products["product_length_cm"] *
    items_products["product_height_cm"] *
    items_products["product_width_cm"]
)
product_summary = (
    items_products
    .groupby("order_id")
    .agg(
        avg_product_weight=("product_weight_g", "mean"),
        avg_product_volume=("product_volume", "mean"),
        unique_categories=("product_category_name", "nunique")
    )
    .reset_index()
)
dfML = dfML.merge(
    product_summary,
    on="order_id",
    how="left"
)

In [13]:
dfML.info()

dfML.head()

dfML.shape

<class 'pandas.DataFrame'>
RangeIndex: 99992 entries, 0 to 99991
Data columns (total 20 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   order_id                       99992 non-null  str    
 1   customer_id                    99992 non-null  str    
 2   order_status                   99992 non-null  str    
 3   order_purchase_timestamp       99992 non-null  str    
 4   order_approved_at              99831 non-null  str    
 5   order_delivered_carrier_date   98199 non-null  str    
 6   order_delivered_customer_date  97005 non-null  str    
 7   order_estimated_delivery_date  99992 non-null  str    
 8   customer_city                  99992 non-null  str    
 9   customer_state                 99992 non-null  str    
 10  review_score                   99224 non-null  float64
 11  payment_value                  99991 non-null  float64
 12  payment_installments           99991 non-null  float64
 1

(99992, 20)

In [14]:
dfML.to_csv("olist_ml_dataset.csv", index=False)

In [15]:
print(dfOrders.columns.tolist())
print(dfCustomers.columns.tolist())
print(dfOrderItems.columns.tolist())
print(dfProducts.columns.tolist())
print(dfPayments.columns.tolist())
print(dfReviews.columns.tolist())
print(dfSellers.columns.tolist())

['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']
['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']


In [16]:
df = pd.read_csv("olist_ml_dataset.csv")

df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_city,customer_state,review_score,payment_value,payment_installments,payment_type,number_of_items,total_product_price,freight_value,avg_product_weight,avg_product_volume,unique_categories
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,sao paulo,SP,4.0,38.71,1.0,credit_card,1.0,29.99,8.72,500.0,1976.0,1.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,barreiras,BA,4.0,141.46,1.0,boleto,1.0,118.70,22.76,400.0,4693.0,1.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,vianopolis,GO,5.0,179.12,3.0,credit_card,1.0,159.90,19.22,420.0,9576.0,1.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,sao goncalo do amarante,RN,5.0,72.20,1.0,credit_card,1.0,45.00,27.20,450.0,6000.0,1.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,santo andre,SP,5.0,28.62,1.0,credit_card,1.0,19.90,8.72,250.0,11475.0,1.0


In [17]:
df.columns.tolist()

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'customer_city',
 'customer_state',
 'review_score',
 'payment_value',
 'payment_installments',
 'payment_type',
 'number_of_items',
 'total_product_price',
 'freight_value',
 'avg_product_weight',
 'avg_product_volume',
 'unique_categories']

In [18]:
df = pd.read_csv("olist_ml_dataset.csv")

df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_city,customer_state,review_score,payment_value,payment_installments,payment_type,number_of_items,total_product_price,freight_value,avg_product_weight,avg_product_volume,unique_categories
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,sao paulo,SP,4.0,38.71,1.0,credit_card,1.0,29.99,8.72,500.0,1976.0,1.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,barreiras,BA,4.0,141.46,1.0,boleto,1.0,118.70,22.76,400.0,4693.0,1.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,vianopolis,GO,5.0,179.12,3.0,credit_card,1.0,159.90,19.22,420.0,9576.0,1.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,sao goncalo do amarante,RN,5.0,72.20,1.0,credit_card,1.0,45.00,27.20,450.0,6000.0,1.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,santo andre,SP,5.0,28.62,1.0,credit_card,1.0,19.90,8.72,250.0,11475.0,1.0


In [19]:
#converting the dates
date_columns = [
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_approved_at",
    "order_estimated_delivery_date"
]


for col in date_columns:
    df[col] = pd.to_datetime(df[col])

In [20]:
#creating delivery days 
df["delivery_days"] = (
    df["order_delivered_customer_date"]
    -
    df["order_purchase_timestamp"]
).dt.days
df["delivery_days"].isnull().sum()
df = df.dropna(
    subset=["delivery_days"]
)

In [21]:
df["approval_time"] = (
    df["order_approved_at"]
    -
    df["order_purchase_timestamp"]
).dt.total_seconds()/3600
df["delivery_delay"] = (
    df["order_delivered_customer_date"]
    -
    df["order_estimated_delivery_date"]
).dt.days

In [22]:
df["purchase_month"] = (
    df["order_purchase_timestamp"]
    .dt.month
)
df["purchase_day"] = (
    df["order_purchase_timestamp"]
    .dt.dayofweek
)

In [23]:
features = [
    "customer_state",
    "customer_city",
    "payment_value",
    "payment_installments",
    "number_of_items",
    "freight_value",
    "avg_product_weight",
    "avg_product_volume",
    "unique_categories",
    "approval_time",
    "purchase_month",
    "purchase_day"
]


X = df[features]

y = df["delivery_days"]

In [24]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [25]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

In [26]:
categorical_features = [
    "customer_state",
    "customer_city"
]

numeric_features = [
    "payment_value",
    "payment_installments",
    "number_of_items",
    "freight_value",
    "avg_product_weight",
    "avg_product_volume",
    "unique_categories",
    "approval_time",
    "purchase_month",
    "purchase_day"
]


preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ],
    remainder="passthrough"
)

In [27]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline

In [28]:
model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            RandomForestRegressor(
                n_estimators=100,
                random_state=42
            )
        )
    ]
)

In [ ]:
model.fit(
    X_train,
    y_train
)

In [31]:
X.isnull().sum().sort_values(ascending=False)

avg_product_weight      16
avg_product_volume      16
approval_time           14
payment_installments     1
payment_value            1
customer_city            0
customer_state           0
freight_value            0
number_of_items          0
unique_categories        0
purchase_month           0
purchase_day             0
dtype: int64

In [32]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

In [34]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

In [35]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numeric_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)

In [36]:
model.fit(
    X_train,
    y_train
)

ValueError: Input X contains NaN.
RandomForestRegressor does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [37]:
print(X_train.isnull().sum())
print(X_train.isnull().sum().sum())

customer_state           0
customer_city            0
payment_value            1
payment_installments     1
number_of_items          0
freight_value            0
avg_product_weight      12
avg_product_volume      12
unique_categories        0
approval_time           12
purchase_month           0
purchase_day             0
dtype: int64
38


In [41]:
numeric_features = ...
categorical_features = ...
preprocessor = ...
model = Pipeline(...)

In [42]:
preprocessor = ColumnTransformer(...)

In [43]:
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestRegressor(
            n_estimators=100,
            random_state=42
        ))
    ]
)

In [44]:
model.fit(X_train, y_train)

InvalidParameterError: The 'transformers' parameter of ColumnTransformer must be an instance of 'list'. Got Ellipsis instead.

In [45]:
import numpy as np

np.isinf(X_train.select_dtypes(include=np.number)).sum()

payment_value           0
payment_installments    0
number_of_items         0
freight_value           0
avg_product_weight      0
avg_product_volume      0
unique_categories       0
approval_time           0
purchase_month          0
purchase_day            0
dtype: int64

In [46]:
X_train.dtypes

customer_state              str
customer_city               str
payment_value           float64
payment_installments    float64
number_of_items         float64
freight_value           float64
avg_product_weight      float64
avg_product_volume      float64
unique_categories       float64
approval_time           float64
purchase_month            int32
purchase_day              int32
dtype: object

In [47]:
print(X_train.isnull().sum())
print(X_train.isnull().sum().sum())

customer_state           0
customer_city            0
payment_value            1
payment_installments     1
number_of_items          0
freight_value            0
avg_product_weight      12
avg_product_volume      12
unique_categories        0
approval_time           12
purchase_month           0
purchase_day             0
dtype: int64
38


In [48]:
print(numeric_features)

Ellipsis


In [49]:
preprocessor = ColumnTransformer(
    ...
)

In [50]:
model = Pipeline(
    ...
)

In [51]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

X_train_num = pd.DataFrame(
    imputer.fit_transform(X_train[numeric_features]),
    columns=numeric_features
)

X_train_num.isnull().sum()

KeyError: Ellipsis

In [1]:
categorical_features = [
    "customer_state",
    "customer_city"
]

numeric_features = [
    "payment_value",
    "payment_installments",
    "number_of_items",
    "freight_value",
    "avg_product_weight",
    "avg_product_volume",
    "unique_categories",
    "approval_time",
    "purchase_month",
    "purchase_day"
]

In [2]:
print(numeric_features)
print(categorical_features)

['payment_value', 'payment_installments', 'number_of_items', 'freight_value', 'avg_product_weight', 'avg_product_volume', 'unique_categories', 'approval_time', 'purchase_month', 'purchase_day']
['customer_state', 'customer_city']


In [3]:
logistic_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            LogisticRegression(
                max_iter=1000
            )
        )
    ]
)

NameError: name 'Pipeline' is not defined

In [4]:
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Evaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

ModuleNotFoundError: No module named 'matplotlib'

In [5]:
import pandas as pd
import numpy as np

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Evaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [6]:
df = pd.read_csv("olist_ml_dataset.csv")

In [7]:
df["satisfied_customer"] = np.where(
    df["review_score"] >= 4,
    1,
    0
)

In [8]:
df["satisfied_customer"] = np.where(
    df["review_score"] >= 4,
    1,
    0
)

In [9]:
drop_columns = [
    "order_id",
    "customer_id",
    "review_score"
]

df_model = df.drop(columns=drop_columns)

In [10]:
X = df_model.drop(
    "satisfied_customer",
    axis=1
)

y = df_model["satisfied_customer"]

In [11]:
numeric_features = X.select_dtypes(
    include=["int64","float64"]
).columns


categorical_features = X.select_dtypes(
    include=["object"]
).columns

C:\Users\dipoa\AppData\Local\Temp\ipykernel_11972\132529773.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(


In [12]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ],
    remainder="passthrough"
)

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [14]:
logistic_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            LogisticRegression(
                max_iter=1000
            )
        )
    ]
)

In [15]:
logistic_model.fit(
    X_train,
    y_train
)

ValueError: Input X contains NaN.
LogisticRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [16]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

In [17]:
numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ]
)

In [18]:
categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

In [19]:
numeric_features = [
    "payment_value",
    "payment_installments",
    "number_of_items",
    "total_product_price",
    "freight_value",
    "avg_product_weight",
    "avg_product_volume",
    "unique_categories"
]


categorical_features = [
    "order_status",
    "customer_state",
    "customer_city",
    "payment_type"
]

In [20]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numeric_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)

In [21]:
from sklearn.linear_model import LogisticRegression


logistic_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            LogisticRegression(
                max_iter=1000
            )
        )
    ]
)

In [22]:
logistic_model.fit(
    X_train,
    y_train
)

c:\Users\dipoa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](17,)","['order_status','order_purchase_timestamp','order_approved_at',..., 'avg_product_weight','avg_product_volume','unique_categories']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,17
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of 

In [23]:
y_pred = logistic_model.predict(X_test)

In [24]:
logistic_model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](17,)","['order_status','order_purchase_timestamp','order_approved_at',..., 'avg_product_weight','avg_product_volume','unique_categories']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,17
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of 

In [25]:
y_pred = logistic_model.predict(X_test)

In [26]:
y_probability = logistic_model.predict_proba(X_test)[:,1]

In [27]:
prediction_results = X_test.copy()

prediction_results["actual_satisfaction"] = y_test

prediction_results["predicted_satisfaction"] = y_pred

prediction_results["satisfaction_probability"] = y_probability

In [28]:
prediction_results["order_id"] = df.loc[
    X_test.index,
    "order_id"
]

In [29]:
prediction_results = prediction_results[
    [
        "order_id",
        "actual_satisfaction",
        "predicted_satisfaction",
        "satisfaction_probability"
    ]
    +
    [
        col for col in prediction_results.columns
        if col not in [
            "order_id",
            "actual_satisfaction",
            "predicted_satisfaction",
            "satisfaction_probability"
        ]
    ]
]

In [30]:
prediction_results.to_csv(
    "customer_satisfaction_predictions.csv",
    index=False
)